# tools

> Every tool an agent is given, and which of them one host can actually support.

In [ ]:
#| default_exp tools

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import json, subprocess, tempfile
from fastcore.test import test_eq, test_fail
from fastcore.funccall import get_schema

In [ ]:
#| export
import functools, json, mimetypes, os, re, threading, uuid
from base64 import b64decode
from pathlib import Path
from fastcore.basics import AttrDict, bind
from fastcore.foundation import L
from fastcore.xtras import detect_mime
from shalya.refactor import SG_LANGS, ast_plan, ast_sub
from shalya.core import (Hit, ERR, MAX_TOOL_CHARS, MAX_HITS, MAX_GREP_HITS, MAX_API, GIT_TOOLS,
                         GIT_READ_TOOLS, GIT_WRITE_TOOLS, WRITE_TOOLS, clip, clip_lines, cmds,
                         edits, apply_edits, diff_text, err, failed, is_write, writes, acts, has_effect,
                         ACTING_TOOLS, summary, summarise, one_line as _1)
from shalya.host import Host, HostError, LocalHost, host_err
from shalya.skills import Skill, find, skill_index

A tool is a typed function whose docstring guides model calls. Each factory binds tools to a host.
`mx` sets the result budget for the calling model.

In [ ]:
#| export
def readable(host, path, must_exist=False):
    "Resolve a path a tool is only going to read. `reading=True` is the read-outside allowance."
    return host.check(path, must_exist=must_exist, reading=True)

All read-only tools resolve paths through `readable` and `check(reading=True)`. The host controls
read access outside its roots.

## Seeing the code

In [ ]:
#| export
def code_tools(host, mx=MAX_TOOL_CHARS):
    "Code search and structure tools."

    @summary(lambda a: f'Search {_1(a.get("query"))}')
    def search_code(query: str) -> str:
        """Search the codebase and installed packages for `query`.
        Uses semantic search when an index is ready and literal search otherwise.
        """
        hits = host.search(query, limit=MAX_HITS)
        if not hits: return f'no matches ({host.search_note})'
        rows = []
        for h in hits:
            target = ('NOTEBOOK -- use this exact path with notebook_cells, then view_cell/edit_cell'
                      if str(h.path).lower().endswith('.ipynb')
                      else 'FILE -- use this exact path with view_file/edit_file')
            rows.append(f'{h.path}:{h.line}  {h.symbol or ""}  {h.text}\n  {target}')
        return clip(f'[{host.search_note}]\n' + '\n'.join(rows), mx)

    @summary(lambda a: f'Similar to {a.get("path","")}:{a.get("line", 1)}')
    def similar_code(path: str, line: int = 1) -> str:
        "Find implementations similar to the function at `path`:`line`."
        hits = host.peers(str(readable(host, path)), int(line), limit=MAX_HITS)
        if not hits: return f'nothing similar ({host.search_note})'
        return clip('\n'.join(f'{h.path}:{h.line}  {h.symbol or ""}  {h.text}' for h in hits), mx)

    @summary(lambda a: f'Outline {a.get("path","")}')
    def outline(path: str) -> str:
        "The defs and classes in one file, with line numbers."
        syms = host.symbols(str(readable(host, path)))
        if not syms: return f'no symbols in {path}'
        return clip('\n'.join(f'{int(getattr(s, "score", 0))*" "}{s.line}: {s.symbol}' for s in syms), mx)

    @summary(lambda a: f'List files {_1(a.get("pattern","")) or "(all)"}')
    def list_files(pattern: str = '') -> str:
        "Files in the open folders, optionally filtered by a substring of the path."
        ps = [str(p) for p in host.walk()]
        if pattern: ps = [p for p in ps if pattern.lower() in p.lower()]
        return clip_lines(ps, n=mx, more='narrow `pattern`', empty='no matching files')

    @summary(lambda a: f'Grep {_1(a.get("pattern",""))}' + (f' in {a["path_filter"]}' if a.get('path_filter') else ''))
    def grep(pattern: str, path_filter: str = '', regex: bool = True, ignore_case: bool = False) -> str:
        """Find matching lines in the open folders.
        `path_filter` is a path substring. Set `regex=False` for literal matching.
        """
        if not str(pattern or '').strip(): return err('grep needs a pattern')
        flags = re.IGNORECASE if ignore_case else 0
        try: rx = re.compile(pattern if regex else re.escape(pattern), flags)
        except re.error as e: return err('bad pattern', e)
        try: fast = host.grep(pattern, path_filter=path_filter, regex=regex, ignore_case=ignore_case, limit=MAX_GREP_HITS)
        except Exception: fast = None
        if fast is not None:
            if not fast: return f'no matches for {pattern!r}'
            capped = len(fast) >= MAX_GREP_HITS
            head = f'{len(fast)}{"+" if capped else ""} match(es)'
            rows = [f'{h.path}:{h.line}: {h.text}' for h in fast]
            return clip_lines([head] + rows, n=mx, more='narrow `pattern` or set `path_filter`')
        pf, hits, scanned, capped = str(path_filter or '').lower(), [], 0, False
        for p in host.walk():
            sp = str(p)
            if pf and pf not in sp.lower(): continue
            try: text = host.read(sp)
            except Exception: continue
            if not text: continue
            scanned += 1
            for i, line in enumerate(text.splitlines(), 1):
                if rx.search(line):
                    hits.append(f'{sp}:{i}: {line.strip()[:200]}')
                    if len(hits) >= MAX_GREP_HITS: capped = True; break
            if capped: break
        if not hits: return f'no matches for {pattern!r} in {scanned} file(s)'
        head = f'{len(hits)}{"+" if capped else ""} match(es) in {scanned} file(s) searched'
        return clip_lines([head] + hits, n=mx, more='narrow `pattern` or set `path_filter`')

    @summary(lambda a: f'List {a.get("path") or "(open folders)"}')
    def ls(path: str = '') -> str:
        "List one directory with subdirectories first and file sizes. Empty `path` lists each open root."
        roots = ([readable(host, path)] if str(path or '').strip() else [host.check(r) for r in host.roots])
        out = []
        for d in roots:
            if not d.exists(): out.append(f'{d}: does not exist'); continue
            if d.is_file(): out.append(f'{d}  ({d.stat().st_size} bytes, a file)'); continue
            try: kids = sorted(d.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
            except Exception as e: out.append(err(f'cannot list {d}', e)); continue
            out.append(f'{d}/')
            for k in kids:
                if k.name.startswith('.') and k.name not in ('.agents', '.leela'): continue
                try: out.append(f'  {k.name}/' if k.is_dir() else f'  {k.name}  {k.stat().st_size}')
                except Exception: out.append(f'  {k.name}')
        return clip_lines(out, n=mx, more='name a subdirectory to list it', empty='(nothing)')

    @summary(lambda a: f'Public API of {a.get("package","?")}')
    def public_api(package: str) -> str:
        "Every public name a package exports, with its docstring and where it is defined."
        if not str(package or '').strip(): return err('public_api needs a package name')
        try: api = host.public_api(package)
        except Exception as e: return err(f'cannot list the API of {package}', e)
        if not api: return (f'nothing public indexed for {package!r}. covers installed packages and open folders, not stdlib')
        rows = [f'{h.symbol}  {h.path}:{h.line}  {h.text}' for h in api]
        return clip_lines([f'{len(rows)} public name(s) in {package}'] + rows, n=mx, more='read one with view_file')

    tools = [search_code, grep, ls, similar_code, outline, list_files]
    #: `indexed` is `LocalHost`'s, not part of the code group's contract, so a host may declare
    #: the group and not carry it. With no index `public_api` could only ever refuse.
    if getattr(host, 'indexed', False): tools.append(public_api)
    return tools

A real folder, a real host, and the tools closed over it. With no Kosha index, `search_code` is
ripgrep and `outline` is the parser.

In [ ]:
root = Path(tempfile.mkdtemp()).resolve()/'proj'
(root/'pkg').mkdir(parents=True)
(root/'pkg'/'sizes.py').write_text('def threshold(n):\n    "Half of n."\n    return n // 2\n')
(root/'pkg'/'use.py').write_text('from .sizes import threshold\n\ndef budget(): return threshold(8192)\n')
host = LocalHost([root], index=False)
ct = {t.__name__: t for t in code_tools(host)}
sorted(ct)

['grep', 'list_files', 'ls', 'outline', 'search_code', 'similar_code']

In [ ]:
test_eq(sorted(ct), ['grep', 'list_files', 'ls', 'outline', 'search_code', 'similar_code'])
assert 'public_api' not in ct, 'no index, so nothing to list a public surface from'
assert 'threshold' in ct['search_code']('threshold')
assert 'sizes.py' in ct['list_files']('sizes')
assert 'threshold' in ct['outline']('pkg/sizes.py')

A no-match result names the search engine. This distinguishes no matches from an unavailable
index.

In [ ]:
ct['search_code']('nonexistent_symbol_xyz')

'no matches (Kosha sync in progress; literal fallback via ripgrep)'

In [ ]:
miss = ct['search_code']('nonexistent_symbol_xyz')
assert 'no matches' in miss.lower(), miss
assert 'ripgrep' in miss or 'fallback' in miss, miss

The fallback above must work before indexing finishes. A second host below builds a real Kosha index over a small package. It checks semantic retrieval, indexed exports and references across files.

In [ ]:
from litesearch import repo_root

In [ ]:
repo_root()
indexed_host = LocalHost([repo_root()], rerank=False)
assert indexed_host.wait_index(180), indexed_host.search_note
indexed_tools = {t.__name__: t for t in code_tools(indexed_host)}; indexed_tools

/Users/71293/code/personal/orgs/shalya/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


parse files from /Users/71293/code/personal/orgs/shalya:   0%|          | 0/6 [00:00<?, ?it/s]

<div><progress max="3" value="0"></progress> </div>

parse files from /Users/71293/code/personal/orgs/shalya: 100%|██████████| 6/6 [00:00<00:00, 281.78it/s]

{'search_code': <function __main__.code_tools.<locals>.search_code(query: str) -> str>,
 'grep': <function __main__.code_tools.<locals>.grep(pattern: str, path_filter: str = '', regex: bool = True, ignore_case: bool = False) -> str>,
 'ls': <function __main__.code_tools.<locals>.ls(path: str = '') -> str>,
 'similar_code': <function __main__.code_tools.<locals>.similar_code(path: str, line: int = 1) -> str>,
 'outline': <function __main__.code_tools.<locals>.outline(path: str) -> str>,
 'list_files': <function __main__.code_tools.<locals>.list_files(pattern: str = '') -> str>,
 'public_api': <function __main__.code_tools.<locals>.public_api(package: str) -> str>}

In [ ]:
indexed_tools['search_code']('code tools')

'[Kosha semantic + keyword index over 1 folder(s) and environment fused with ripgrep]\n/Users/71293/code/personal/orgs/shalya/shalya/tools.py:785  shalya.tools.tools_for  def tools_for(host, get_skills=None, extra=(), mx=MAX_TOOL_CHARS, drop=(), image=None): """Every tool this host declares it can support, plus whatever else was registered. `image` is a built image group, or None. It needs to know what the t\n  FILE -- use this exact path with view_file/edit_file\n/Users/71293/code/personal/orgs/shalya/nbs/02_tools.ipynb:524    "\'[Kosha semantic + keyword index over 1 folder(s) and environment fused with ripgrep]\\\\n/Users/71293/code/personal/orgs/shalya/nbs/02_tools.ipynb:473    \\"\\\\\'[Kosha semantic + keyword index over 1 fo\n  NOTEBOOK -- use this exact path with notebook_cells, then view_cell/edit_cell\n/Users/71293/code/personal/orgs/rishi/.venv/lib/python3.13/site-packages/matplotlib/backend_tools.py:982  matplotlib.backend_tools.add_tools_to_container  def add_tools_to_cont

In [ ]:
indexed_tools['public_api']('shalya.host')

"125 public name(s) in shalya.host\nshalya.host.ApiHost  /Users/71293/code/personal/orgs/shalya/shalya/host.py:311  Reading an API specification and calling what it describes.\nshalya.host.AskHost  /Users/71293/code/personal/orgs/shalya/shalya/host.py:184  Model-backed answers from memory.\nshalya.host.Capability  /Users/71293/code/personal/orgs/shalya/shalya/host.py:21  One capability group. A host declares the group by inheriting the class that names it.\nshalya.host.CodeHost  /Users/71293/code/personal/orgs/shalya/shalya/host.py:87  Code search and structure.\nshalya.host.GitHost  /Users/71293/code/personal/orgs/shalya/shalya/host.py:312  Git operations on a working tree inside `roots`.\nshalya.host.Host  /Users/71293/code/personal/orgs/shalya/shalya/host.py:27  The application under an agent: the folders it may touch, and what it declares it can do.\nshalya.host.LocalHost  /Users/71293/code/personal/orgs/shalya/shalya/host.py:403  Reference host for local folders.\nshalya.host.Memo

In [ ]:
indexed_tools['similar_code']('shalya/host.py', 32)

'/Users/71293/code/personal/orgs/shalya/shalya/host.py:27  shalya.host.Host  class Host(ABC): "The application under an agent: the folders it may touch, and what it declares it can do." group = \'file\' #: every host has the path boundary the file tools need without = frozenset() #: groups this instance cannot do, wha\n/Users/71293/code/personal/orgs/shalya/shalya/core.py:34  shalya.core.host_err  def host_err(e): "A caught exception, for a user-facing surface." return f\'{type(e).__name__}: {e}\'\n/Users/71293/code/personal/orgs/shalya/README.md:16    from shalya.host import LocalHost\n/Users/71293/code/personal/orgs/shalya/shalya/tools.py:785  shalya.tools.tools_for  def tools_for(host, get_skills=None, extra=(), mx=MAX_TOOL_CHARS, drop=(), image=None): """Every tool this host declares it can support, plus whatever else was registered. `image` is a built image group, or None. It needs to know what the t\n/Users/71293/code/personal/orgs/shalya/README.md:23    [`LocalHost`](https://ved

## Files

`view_file` returns `lineno|hash|content`. Edits must quote a current address. A stale hash is
refused, preventing edits to unseen content.

In [ ]:
#| export
def file_tools(host, mx=MAX_TOOL_CHARS):
    "Reading and editing files, by exact text or by hash-verified address."

    @writes
    @summary(lambda a: f'Open folder {a.get("path","")}')
    def add_root(path: str) -> str:
        """Add an existing folder to the read and write boundary.
        The user must name the folder. This write requires approval.
        """
        try: return f'opened {host.add_root(path)}. Open folders: ' + ', '.join(host.roots)
        except Exception as e: return err(f'could not open {path}', e)

    @summary(lambda a: f'View {a.get("path","")}' + (f':{a.get("start","")}-{a.get("end","")}' if a.get('start') or a.get('end') else ''))
    def view_file(path: str, start: int = 0, end: int = 0) -> str:
        """Read `path` as `lineno|hash|content` lines.
        The hashes are addresses for `edit_file`. `start` and `end` limit the line range.
        """
        from exhash import lnhashview, lnhashview_file
        p = readable(host, path)
        try: text = host.read(str(p))
        except Exception: text = None
        if text is None and not p.exists(): return err(f'no such file: {p}')
        view = str(lnhashview(text, start or None, end or None) if text is not None else lnhashview_file(str(p), start or None, end or None))
        return clip_lines(view.splitlines(), start=(start or 1), n=mx,
                          more='call view_file(path, start={next}) to continue')

    @writes
    @summary(lambda a: f'Edit {a.get("path","")}')
    def replace_text(path: str, spec: str) -> str:
        """Apply exact-text replacements and return the diff.
        `spec` is a JSON array of `oldText` and `newText` objects. Each non-empty `oldText` must occur once in the current file. Edits cannot overlap. A rejected edit writes nothing. Use `create_file` for new files.
        """
        p = host.check(path)
        if hasattr(host, 'check_write'): host.check_write(p)
        try: items = edits(spec)
        except Exception as e: return err('could not parse edits', e)
        if not items: return err('no edits given')
        try: before = host.read(str(p))
        except Exception as e: return err(f'could not read {p}', e)
        if before is None: return err(f'no such file: {p}. Use create_file to create it')
        try: after = apply_edits(before, items)
        except ValueError as e: return err(str(e))
        if after == before: return err('the edits changed nothing; check oldText against a fresh view_file')
        try: host.write(str(p), after)
        except Exception as e: return err('write failed', e)
        return clip(f'replaced {len(items)} block(s) in {p}\n' + diff_text(before, after, str(p)), mx)

    @writes
    @summary(lambda a: f'Edit {a.get("path","")}')
    def edit_file(path: str, commands: str) -> str:
        """Apply hash-verified exhash `commands` and return the diff.
        Each command starts with an address from `view_file`. The tool writes only after all commands succeed.
        """
        from exhash import file_exhash
        p = host.check(path)
        if hasattr(host, 'check_write'): host.check_write(p)
        try: cs = cmds(commands)
        except Exception as e: return err('could not parse commands', e)
        if not cs: return err('no commands given')
        try: return clip(str(file_exhash(str(p), *cs)), mx)
        except Exception as e: return err('edit failed', e)

    @writes
    @summary(lambda a: f'Create {a.get("path","")}')
    def create_file(path: str, text: str = '') -> str:
        "Create (or overwrite) a whole file. For changes to an existing file prefer `replace_text`."
        try: return f'wrote {host.write(path, text)}'
        except Exception as e: return err('write failed', e)

    @writes
    @summary(lambda a: f'Structural edit {a.get("path","") or "every open folder"}')
    def ast_edit(pattern: str, replacement: str, path: str = '') -> str:
        """Rewrite Python by shape rather than by text, and return the diff.
        `pattern` and `replacement` are ast-grep patterns: `area($A)` matches every call to `area` whatever its argument, and `$A` in the replacement is what that call passed. Use `$$$A` where the count varies. Never matches inside a comment or a string, or a longer name that contains yours, which is what makes it safe for a rename across files. Leave `path` empty to change every Python file in the open folders.
        A definition is not renameable this way: every pattern matching one matches its whole body. Rename the calls and the imports here, and change the `def` with `replace_text`.
        """
        if path:
            try:
                p = host.check(path)
                if hasattr(host, 'check_write'): host.check_write(p)
            except Exception as e: return err(f'cannot write {path}', e)
            try: before = host.read(str(p))
            except Exception as e: return err(f'could not read {p}', e)
            if before is None: return err(f'no such file: {p}')
            try: after, n = ast_sub(str(p), before, pattern, replacement)
            except ValueError as e: return err(str(e))
            if not n: return err(f'{pattern!r} matched nothing in {p}')
            try: host.write(str(p), after)
            except Exception as e: return err('write failed', e)
            return clip(f'{n} match(es) in {p}\n' + diff_text(before, after, str(p)), mx)
        try: rows, _ = ast_plan(host, pattern, replacement)
        except ValueError as e: return err(str(e))
        if not rows: return err(f'{pattern!r} matched nothing in ' + ', '.join(host.roots))
        done, out = [], []
        for r in rows:
            if hasattr(host, 'check_write'):
                try: host.check_write(Path(r.path))
                except Exception as e: return err(f'{r.path} cannot be written', e)
        for r in rows:
            try: host.write(r.path, r.after)
            except Exception as e: return err(f'wrote {len(done)} file(s), then {r.path} failed', e)
            done.append(r)
            out.append(diff_text(r.before, r.after, r.path))
        return clip(f'{sum(r.edits for r in done)} match(es) in {len(done)} file(s)\n' + '\n'.join(out), mx)

    return [view_file, replace_text, edit_file, create_file, ast_edit, add_root]

All commands succeed or nothing is written.

In [ ]:
ft = {t.__name__: t for t in file_tools(host)}
p = root/'pkg'/'greet.py'
p.write_text('def greet(name):\n    return "hello " + name\n')
print(ft['view_file'](str(p)))

1|2337|def greet(name):
2|709e|    return "hello " + name


In [ ]:
line2 = ft['view_file'](str(p)).splitlines()[1]
addr = line2.split('|')[0] + '|' + line2.split('|')[1] + '|'
ft['edit_file'](str(p), json.dumps([[addr, 's', 'hello', 'howdy']]))
test_eq(p.read_text(), 'def greet(name):\n    return "howdy " + name\n')

In [ ]:
stale = ft['edit_file'](str(p), json.dumps([['2|0000|', 's', 'howdy', 'hi']]))
assert failed(stale), stale
test_eq(p.read_text(), 'def greet(name):\n    return "howdy " + name\n')

`replace_text` is the same write through the exact-text vocabulary, and it refuses for the same
reasons `apply_edits` refuses.

In [ ]:
ft['replace_text'](str(p), json.dumps([{'oldText': 'howdy', 'newText': 'hey'}]))
test_eq(p.read_text(), 'def greet(name):\n    return "hey " + name\n')
assert failed(ft['replace_text'](str(p), json.dumps([['name', 'who']])))   # matches twice

Every write tool carries its own mark, and the frozenset of names says the same thing for a caller
that only has a name.

In [ ]:
#: A structural rename reaches every call in the file and nothing that merely reads like one.
q = root/'pkg'/'calls.py'
q.write_text('print(greet(1))\ngreet_all = 2\ns = "greet(3)"\n')
ft['ast_edit']('greet($A)', 'hail($A)', str(q))
test_eq(q.read_text(), 'print(hail(1))\ngreet_all = 2\ns = "greet(3)"\n')
assert failed(ft['ast_edit']('greet($A)', 'hail($B)', str(q)))      # $B is not captured
assert failed(ft['ast_edit']('nothing($A)', 'x($A)', str(q)))       # nothing matched, nothing written
test_eq(q.read_text(), 'print(hail(1))\ngreet_all = 2\ns = "greet(3)"\n')

In [ ]:
writers = {t.__name__ for t in file_tools(host) if is_write(t)}
test_eq(writers, {'edit_file', 'replace_text', 'ast_edit', 'create_file', 'add_root'})
assert writers <= WRITE_TOOLS, writers - WRITE_TOOLS

## Notebooks

In [ ]:
#| export
def notebook_tools(host, mx=MAX_TOOL_CHARS):
    "Notebooks, addressed by cell id rather than by line."

    @summary(lambda a: f'Cells of {a.get("path","")}')
    def notebook_cells(path: str) -> str:
        "List a notebook's cells: id, type, and first line. Cell ids are what `edit_cell` addresses."
        try: rows = host.nb_cells(str(readable(host, path)))
        except NotImplementedError: raise
        except Exception as e: return err('could not read notebook', e)
        return clip('\n'.join(f'{i}  {t:8} {(s or "").strip().splitlines()[0][:100] if (s or "").strip() else ""}' for i, t, s in rows) or '(empty notebook)')

    @summary(lambda a: f'View {a.get("path","")} cell {a.get("cell_id","?")}')
    def view_cell(path: str, cell_id: str) -> str:
        "Read one notebook cell as `lineno|hash|content` lines, ready to address with `edit_cell`."
        from exhash import lnhashview_cell
        try: return clip(str(lnhashview_cell(str(readable(host, path)), cell_id)))
        except Exception as e: return err('could not read cell', e)

    @writes
    @summary(lambda a: f'Edit {a.get("path","")} cell {a.get("cell_id","?")}')
    def edit_cell(path: str, cell_id: str, commands: str) -> str:
        "Edit one notebook cell's source with exhash commands from `view_cell`. Same format as `edit_file`."
        from exhash import cell_exhash
        try: cs = cmds(commands)
        except Exception as e: return err('could not parse commands', e)
        try: return clip(str(cell_exhash(str(host.check(path)), cell_id, *cs)))
        except Exception as e: return err('edit failed', e)

    @writes
    @summary(lambda a: f'Add {a.get("cell_type","code")} cell to {a.get("path","")}')
    def add_cell(path: str, source: str, index: int = -1, cell_type: str = 'code') -> str:
        "Insert a new cell into a notebook at `index` (-1 appends). Creates the notebook if needed."
        try: return f'added cell {host.nb_add_cell(str(host.check(path)), source, int(index), cell_type)} to {path}'
        except NotImplementedError: raise
        except Exception as e: return err('could not add cell', e)

    return [notebook_cells, view_cell, edit_cell, add_cell]

The two operations that need to know what a notebook is, and the two that only need a cell id.

In [ ]:
nt = {t.__name__: t for t in notebook_tools(host)}
said = nt['add_cell']('nb/demo.ipynb', 'x = 1')
cid = said.split()[2]
said, nt['notebook_cells']('nb/demo.ipynb')

('added cell 75ea04d7 to nb/demo.ipynb', '75ea04d7  code     x = 1')

In [ ]:
assert cid in nt['notebook_cells']('nb/demo.ipynb')
assert 'x = 1' in nt['view_cell']('nb/demo.ipynb', cid)
line = nt['view_cell']('nb/demo.ipynb', cid).splitlines()[0]
nt['edit_cell']('nb/demo.ipynb', cid, json.dumps([['|'.join(line.split('|')[:2]) + '|', 's', 'x = 1', 'x = 2']]))
assert 'x = 2' in nt['view_cell']('nb/demo.ipynb', cid), nt['view_cell']('nb/demo.ipynb', cid)

## The web, and what was read before

Two groups, and the split is the point. The web tools go out now. The memory tools recall what going
out already found, without going out again.

In [ ]:
#| export
def web_tools(host, mx=MAX_TOOL_CHARS):
    "The web, for the questions whose answer depends on current documentation."

    @summary(lambda a: f'Web search: {_1(a.get("query"))}')
    def web_search(query: str) -> str:
        "Search the web and return titles and URLs."
        docs = host.web_search(query, n=MAX_HITS)
        if not docs: return f'no results ({host.research_note})'
        return clip('\n'.join(f'{d.title}\n  {d.url}' for d in docs))

    @summary(lambda a: f'Web fetch: {_1(a.get("url"), 120)}')
    def read_url(url: str, remember: bool = True) -> str:
        """Read `url` as source-appropriate text.
        Results enter durable memory unless `remember=False`.
        """
        d = host.read_url(url, remember=remember)
        return clip(d.text if d else f'could not read {url} ({host.research_note})')

    def no_save_read(url: str) -> str:
        "Read one web page without saving it to durable memory."
        return read_url(url, False)
    no_save_read.__name__ = 'read_url'
    read_url.read_only = no_save_read

    @acts
    @summary(lambda a: f'Research: {_1(a.get("query"))}')
    def research(query: str) -> str:
        "Search the web and read the top results into a cited digest."
        return clip(host.research(query) or f'nothing found ({host.research_note})')

    return [web_search, read_url, research]

A result provider isolates the tool contract. These checks cover model-facing text and argument forwarding; `host` covers fossick itself.

In [ ]:
class WebResults:
    research_note = 'fixture'
    def web_search(self, query, n=20):
        return [AttrDict(title='Python', url='https://python.org')] if query else []
    def read_url(self, url, remember=True):
        self.remember = remember
        return AttrDict(text='# Python', url=url)
    def research(self, query): return 'Python is a programming language.'

web_results = WebResults()
wt = {t.__name__: t for t in web_tools(web_results)}

In [ ]:
test_eq(sorted(wt), ['read_url', 'research', 'web_search'])
test_eq(wt['web_search']('python'), 'Python\n  https://python.org')
test_eq(wt['web_search'](''), 'no results (fixture)')
test_eq(wt['read_url']('https://python.org', remember=False), '# Python')
test_eq(web_results.remember, False)
test_eq(wt['research']('python'), 'Python is a programming language.')
assert not any(is_write(t) for t in wt.values())

In [ ]:
#| export
def memory_tools(host, mx=MAX_TOOL_CHARS):
    "Durable pages and research recalled as document sections rather than flat snippets."

    @summary(lambda a: f'Memory search: {_1(a.get("query"))}')
    def memory_search(query: str, limit: int = 8) -> str:
        "Search remembered pages and return matching sections with breadcrumbs."
        try: return clip(json.dumps(list(host.memory_search(query, int(limit))), default=str), MAX_TOOL_CHARS * 2)
        except Exception as e: return err('memory search failed', e)

    @summary(lambda a: f'Memory tree {_1(a.get("document","")) or "(everything)"}')
    def memory_tree(document: str = '') -> str:
        """Browse remembered document headings.
        `document` is a title substring or document id. An empty value lists all roots.
        """
        try: return clip(json.dumps(host.memory_tree(document), default=str), MAX_TOOL_CHARS * 2)
        except Exception as e: return err('memory tree failed', e)

    @summary(lambda a: f'Memory read {a.get("node_id","?")}')
    def memory_read(node_id: str) -> str:
        "Read one whole remembered section by the node id returned by memory_search/tree."
        try: return clip(json.dumps(host.memory_read(node_id), default=str), MAX_TOOL_CHARS * 3)
        except Exception as e: return err('memory read failed', e)

    @summary(lambda a: 'Map remembered topics')
    def memory_topics(limit: int = 12) -> str:
        "Map remembered material into labelled semantic clusters and representative members."
        try: return clip(json.dumps(host.memory_topics(int(limit)), default=str), MAX_TOOL_CHARS * 2)
        except Exception as e: return err('memory topics failed', e)

    @writes
    @summary(lambda a: f'Forget {a.get("doc_id","?")}')
    def memory_forget(doc_id: str) -> str:
        """Delete a remembered document and its index data.
        Call only at the user's request.
        """
        try: return 'forgot document' if host.memory_forget(doc_id) else 'document was not forgotten'
        except Exception as e: return err('memory purge failed', e)

    return [memory_search, memory_tree, memory_read, memory_topics, memory_forget]

In [ ]:
m=memory_tools(host)
summarise(m[-2])

'Map remembered topics'

In [ ]:
#| export
def watch_tools(host, mx=MAX_TOOL_CHARS):
    "Standing interests: what to put back on the desk later, and what has come due now."

    @summary(lambda a: f'Remember {_1(a.get("title") or a.get("text"))}')
    def remember(text: str, title: str = '', tags: str = '') -> str:
        """Store a conclusion in durable memory.
        `tags` is a comma-separated list.
        """
        try:
            d = host.remember(text, title=title or None, tags=[t.strip() for t in tags.split(',') if t.strip()])
            return f"remembered {d.get('title')!r} as {d.get('doc_id')}"
        except Exception as e: return err('could not remember', e)

    @acts
    @summary(lambda a: f'Remind every {a.get("every","1w")}: {_1(a.get("text"))}')
    def set_reminder(text: str, every: str = '1w', note: str = '') -> str:
        """Store `text` in memory every `every`.
        Due reminders appear in `memory_search` and `poll_watches`.
        """
        try:
            w = host.watch(text, action='remind', every=every, note=note or None)
            return f"reminder {w['id']} set, every {every}"
        except Exception as e: return err('could not set reminder', e)

    @acts
    @summary(lambda a: f'Watch {_1(a.get("url"), 100)} every {a.get("every","1d")}')
    def watch_url(url: str, every: str = '1d', note: str = '') -> str:
        "Read `url` every `every` and store each version in memory."
        try:
            w = host.watch(url, action='url', every=every, note=note or None)
            return f"watching {url} as {w['id']}, every {every}"
        except Exception as e: return err('could not watch', e)

    @summary(lambda a: 'List watches')
    def list_watches(due_only: bool = False) -> str:
        "List watches and reminders, soonest first. `due_only` returns due entries."
        try:
            ws = host.watches(due_only=bool(due_only))
            if not ws: return 'nothing is being watched'
            return clip('\n'.join(
                f"{w['id']}  {w['action']:8} every {int(w['every'])}s  runs={w['runs']}"
                f"  {w.get('last_status') or 'never run'}  {str(w['target'])[:80]}" for w in ws))
        except Exception as e: return err('could not list watches', e)

    @writes
    @summary(lambda a: f'Cancel watch {a.get("watch_id","?")}')
    def cancel_watch(watch_id: str) -> str:
        "Delete a watch by id at the user's request."
        try:
            host.unwatch(watch_id)
            return f'cancelled {watch_id}'
        except Exception as e: return err('could not cancel', e)

    @summary(lambda a: 'Poll watches')
    def poll_watches() -> str:
        "Run due watches and report what entered memory."
        try:
            r = host.poll()
            if not r.get('ran'): return f"nothing due ({r.get('checked', 0)} watched)"
            lines = [f"{x['status']:7} {x['action']:8} {str(x['target'])[:90]}" for x in r['results']]
            return clip(f"{r['ran']} of {r['checked']} fired\n" + '\n'.join(lines))
        except Exception as e: return err('poll failed', e)

    return [remember, set_reminder, watch_url, list_watches, cancel_watch, poll_watches]

A watch is registered, listed, polled and cancelled, and a reminder is a watch that files its own
text.

Answering out of memory is its own group. It takes a model, and a host can remember pages and
search them without having anything to ask them with.

In [ ]:
#| export
def ask_tools(host, mx=MAX_TOOL_CHARS):
    "Model-backed answers from memory."

    @summary(lambda a: f'Ask memory: {_1(a.get("question"))}')
    def ask_memory(question: str, document: str = '', instruction: str = '') -> str:
        """Answer a question from remembered research with citations.
        `document` limits the source. Private sources withhold identifying details. Use `instruction` to request a count, total, comparison, or yes/no answer. Never request withheld details.
        """
        try: r = host.ask(question, ref=document or None, instruction=instruction)
        except NotImplementedError: raise
        except Exception as e: return err('could not ask memory', e)
        rows = [str(r.get('answer') or '(no answer)')]
        if (p := r.get('pii')) and p.get('has_pii'):
            rows.append(f"\n[answered on a local model; it holds back "
                        f"{', '.join(sorted(p.get('identifying') or {}))}. Reply with `instruction=` "
                        f"to say what you need -- a count, a total, a comparison, a yes or no.]")
        if (c := r.get('cited')):
            rows.append('\n' + '\n'.join(f"[{x['n']}] {x['breadcrumb']}  ({x['node_id']})" for x in c))
        return clip('\n'.join(rows), MAX_TOOL_CHARS * 2)

    return [ask_memory]

A deterministic result provider covers JSON rendering, citations, watch summaries, errors and write marks. `host` covers storage adapters.

In [ ]:
class MemoryResults:
    def memory_search(self, query, limit=8):
        if query == 'fail': raise RuntimeError('vault unavailable')
        return [{'id': 'd1', 'title': 'Kettles', 'text': 'boils at 100C'}][:limit]
    def memory_tree(self, document=''): return [{'id': 'd1', 'title': 'Kettles'}]
    def memory_read(self, node_id): return {'id': node_id, 'text': 'boils at 100C'}
    def memory_topics(self, limit=12): return [{'topic': 'home'}][:limit]
    def memory_forget(self, doc_id): return doc_id == 'd1'
    def remember(self, text, title=None, tags=()): return {'doc_id': 'd2', 'title': title or text}
    def watch(self, target, action='remind', every='1d', note=None):
        return {'id': 'w1', 'target': target, 'action': action, 'every': 86400, 'runs': 0}
    def watches(self, due_only=False):
        return [] if due_only else [{'id': 'w1', 'target': 'renew domain', 'action': 'remind',
                                     'every': 86400, 'runs': 0, 'last_status': None}]
    def unwatch(self, watch_id): return watch_id == 'w1'
    def poll(self):
        return {'ran': 1, 'checked': 1,
                'results': [{'status': 'ok', 'action': 'remind', 'target': 'renew domain'}]}
    def ask(self, question, ref=None, instruction=''):
        return {'answer': 'Water boils at 100C.',
                'cited': [{'n': 1, 'breadcrumb': 'Kettles', 'node_id': 'd1'}]}

memory_results = MemoryResults()
mt = {t.__name__: t for t in memory_tools(memory_results)}
wa = {t.__name__: t for t in watch_tools(memory_results)}
at = {t.__name__: t for t in ask_tools(memory_results)}

In [ ]:
test_eq(sorted(mt), ['memory_forget', 'memory_read', 'memory_search', 'memory_topics', 'memory_tree'])
test_eq(sorted(wa), ['cancel_watch', 'list_watches', 'poll_watches', 'remember', 'set_reminder', 'watch_url'])
test_eq(sorted(at), ['ask_memory'])
assert 'boils at 100C' in mt['memory_search']('kettle')
assert failed(mt['memory_search']('fail')) and 'vault unavailable' in mt['memory_search']('fail')
test_eq(mt['memory_forget']('d1'), 'forgot document')
test_eq(wa['remember']('fact', title='Facts', tags='one, two'), "remembered 'Facts' as d2")
test_eq(wa['set_reminder']('renew domain'), 'reminder w1 set, every 1w')
assert 'every 86400s' in wa['list_watches']() and 'renew domain' in wa['list_watches']()
test_eq(wa['poll_watches'](), '1 of 1 fired\nok      remind   renew domain')
test_eq(wa['cancel_watch']('w1'), 'cancelled w1')
answer = at['ask_memory']('boiling point')
assert 'Water boils at 100C.' in answer and '[1] Kettles  (d1)' in answer
test_eq({t.__name__ for t in mt.values() if is_write(t)}, {'memory_forget'})
test_eq({t.__name__ for t in wa.values() if is_write(t)}, {'cancel_watch'})

## The live session and the shell

In [ ]:
#| export
def session_tools(host, mx=MAX_TOOL_CHARS):
    "The live kernel the user is working in, and the terminal they are looking at."

    @summary(lambda a: 'List variables')
    def list_vars() -> str:
        "List the variables visible in the user's live session: name, type, and a short value."
        return clip(host.list_vars() or '(empty session)')

    @writes
    @summary(lambda a: f'Run python: {_1((a.get("code") or "").strip().splitlines()[0] if a.get("code") else "")}')
    def run_python(code: str) -> str:
        """Run Python in the user's live namespace.
        Bind results to new names. Mutating or deleting user variables is refused.
        """
        try: return clip(host.run_python(code))
        except NotImplementedError: raise
        except Exception as e: return err('run failed', e)

    @acts
    @summary(lambda a: f'Inspect: {_1((a.get("code") or "").strip().splitlines()[0] if a.get("code") else "")}' + ('' if (a.get('scope') or 'isolated') == 'isolated' else f'  [{a["scope"]}]'))
    def inspect_python(code: str, scope: str = 'isolated') -> str:
        """Inspect live variables without changing them.
        `isolated` runs allowlisted Python on a copy. `overlay` permits library calls and stores new names in a private layer. Neither scope can mutate user variables. Use `run_python` to write to the user's namespace.
        """
        try: return clip(host.inspect_python(code, scope=scope))
        except NotImplementedError: raise
        except Exception as e: return err('inspection failed', e)

    @summary(lambda a: 'Read terminal')
    def read_terminal(lines: int = 200) -> str:
        "Read recent IDE terminal output without running a command."
        return clip(host.terminal_text(int(lines)) or 'the terminal has printed nothing yet')

    # No tool per recipe: `run_python` composes one in a line. See `coding_patterns`.
    return [list_vars, run_python, inspect_python, read_terminal]

In [ ]:
#| export
def shell_tools(host, mx=MAX_TOOL_CHARS):
    "Shell command tools."

    @writes
    @summary(lambda a: f'Run {_1(a.get("command"), 110)}')
    def run_shell(command: str, cwd: str = '', timeout: int = 120) -> str:
        """Run one terminating project command and return its exit code and output.
        `cwd` must be in the open folders. `timeout` kills expired commands. Do not start servers, watchers, or REPLs. Use the project's documented commands. One call may require approval.
        """
        cmd = str(command or '').strip()
        if not cmd: return err('no command given')
        try: code, out = host.run_cmd(cmd, cwd=(str(cwd).strip() or None), timeout=int(timeout))
        except NotImplementedError: raise
        except Exception as e: return err('command could not be run', e)
        head = f'exit {code}' + ('' if code == 0 else '  (command FAILED)')
        body = clip((out or '').rstrip() or '(no output)', mx - 200,
                    more='re-run narrowing the command (a single test, `| tail -50`) rather than repeating it')
        return f'{head}\n{body}' if code == 0 else f'{ERR}{head}\n{body}'

    return [run_shell]

The session tools reach the live namespace, and the shell tool reaches the machine. Both are write
tools, and both come back as text rather than raising.

In [ ]:
st = {t.__name__: t for t in session_tools(host)}
sh = {t.__name__: t for t in shell_tools(host)}
st['run_python']('total = 6 * 7'), st['run_python']('total'), sh['run_shell']('echo hi')

('(no output)', '42', 'exit 0\nhi')

In [ ]:
test_eq(st['run_python']('total'), '42')
assert 'total' in st['list_vars']()
assert 'hi' in sh['run_shell']('echo hi')
host.note('a status line')
assert 'a status line' in st['read_terminal'](), st['read_terminal']()
assert is_write(st['run_python']) and is_write(sh['run_shell'])
assert not is_write(st['inspect_python']) and not is_write(st['read_terminal'])
test_eq(sorted(st), ['inspect_python', 'list_vars', 'read_terminal', 'run_python'])

## An API specification

In [ ]:
#| export
def api_tools(host, mx=MAX_TOOL_CHARS):
    "Read an API specification, browse what it declares, and call one operation."

    @summary(lambda a: f'Load API spec {_1(a.get("src"), 80)}')
    def api_load(src: str, name: str = '') -> str:
        "Load an OpenAPI or discovery document and return its operation count and groups."
        try: return clip(json.dumps(host.api_load(src, name), default=str), mx)
        except Exception as e: return err('could not load the spec', e)

    @summary(lambda a: 'API operations' + (f' in {a["group"]}' if a.get('group') else '') + (f' matching {_1(a["match"], 40)}' if a.get('match') else ''))
    def api_ops(group: str = '', name: str = '', match: str = '', offset: int = 0) -> str:
        """List loaded API operations and signatures.
        Filter with `group` or `match`. Use `offset` for later pages.
        """
        try:
            rows = host.api_ops(group, name, match, offset=offset)
            total = host.api_count(group=group, name=name, match=match)
            out = {'operations': rows}
            if offset or len(rows) < total:
                out |= {'matched': total, 'showing': f'{offset + 1}-{offset + len(rows)}',
                        'more': f'call again with offset={offset + len(rows)}'} if offset + len(rows) < total else {'matched': total}
            return clip(json.dumps(out, default=str), mx)
        except Exception as e: return err('could not read the operations', e)

    @acts
    @summary(lambda a: f'API call {a.get("operation","?")}')
    def api_call(operation: str, name: str = '', params: dict = None) -> str:
        "Call `operation` with the declared `params`. Undeclared parameters are errors."
        try: return clip(json.dumps(host.api_call(operation, name, **(params or {})), default=str), mx)
        except Exception as e: return err(f'{operation} failed', e)

    return [api_load, api_ops, api_call]

A deterministic result provider covers JSON rendering, pagination, parameter forwarding and errors. `host` covers API backends.

In [ ]:
class ApiResults:
    def api_load(self, src, name=''): return {'name': name or 'petstore', 'operations': 3}
    def api_ops(self, group='', name='', match='', offset=0):
        rows = [{'name': 'listPets'}, {'name': 'getPet'}, {'name': 'createPet'}]
        if match: rows = [r for r in rows if match.lower() in r['name'].lower()]
        return rows[offset:offset + 1]
    def api_count(self, group='', name='', match=''):
        return len([n for n in ('listPets', 'getPet', 'createPet') if not match or match.lower() in n.lower()])
    def api_call(self, operation, name='', **params):
        if operation == 'fail': raise RuntimeError('service unavailable')
        return {'called': operation, 'with': params}

api_results = ApiResults()
apit = {t.__name__: t for t in api_tools(api_results)}

In [ ]:
test_eq(sorted(apit), ['api_call', 'api_load', 'api_ops'])
test_eq(json.loads(apit['api_load']('spec.json')), {'name': 'petstore', 'operations': 3})
page = json.loads(apit['api_ops']())
test_eq(page['operations'], [{'name': 'listPets'}])
test_eq((page['matched'], page['showing'], page['more']), (3, '1-1', 'call again with offset=1'))
test_eq(json.loads(apit['api_ops'](match='get'))['operations'], [{'name': 'getPet'}])
test_eq(json.loads(apit['api_call']('getPet', params={'id': 7})),
        {'called': 'getPet', 'with': {'id': 7}})
assert failed(apit['api_call']('fail')) and 'service unavailable' in apit['api_call']('fail')

## Skills as tools

In [ ]:
#| export
def skill_tools(host, get_skills, mx=MAX_TOOL_CHARS):
    "Reading discovered skills and creating project-local Agent Skills."

    @summary(lambda a: f'Read skill {a.get("name","?")}')
    def read_skill(name: str) -> str:
        "Read a discovered skill in full before doing the work it covers."
        ss = get_skills()
        s = find(ss, name)
        if s is None:
            return f'no skill matching {name!r}. Available: ' + ', '.join(x.name for x in ss)
        return clip(f'<skill name="{s.name}" from="{s.where}">\n{s.text()}\n</skill>', MAX_TOOL_CHARS * 3)

    @writes
    @summary(lambda a: f'Create skill {a.get("name","?")}')
    def create_skill(name: str, description: str, instructions: str) -> str:
        """Create `.agents/skills/NAME/SKILL.md` without overwriting an existing skill.
        `name` must use lowercase kebab-case. `description` states when the skill applies. `instructions` is its Markdown body. Run `/reload` to advertise the new skill.
        """
        from pathlib import Path
        name = str(name or '').strip()
        if not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*', name):
            return 'skill name must be lowercase kebab-case (for example, notebook-tests)'
        if not str(description or '').strip(): return 'skill description is required'
        if not str(instructions or '').strip(): return 'skill instructions are required'
        roots = list(host.roots or ())
        if not roots: return 'open a project folder before creating a skill'
        target = Path(host.check(Path(roots[0])/'.agents'/'skills'/name/'SKILL.md'))
        exists = target.exists()
        if not exists:
            try: exists = host.read(str(target)) is not None
            except Exception: pass
        if exists: return f'refusing to overwrite existing skill: {target}'
        title = json.dumps(name, ensure_ascii=False)
        desc = json.dumps(' '.join(str(description).split()), ensure_ascii=False)
        text = f'---\nname: {title}\ndescription: {desc}\n---\n\n{str(instructions).strip()}\n'
        try: host.write(str(target), text)
        except Exception as e: return err('could not create skill', e)
        return f'created {target}; run /reload to load it into the current agent'

    return [read_skill, create_skill]

The system prompt lists skill names and descriptions. `read_skill` fetches a body when the agent
needs it.

In [ ]:
from shalya.skills import discover

In [ ]:
skill_path = root/'.agents'/'skills'/'house-style'/'SKILL.md'
skill_path.parent.mkdir(parents=True)
skill_path.write_text(
    '---\nname: house-style\ndescription: How code is written here.\n---\n\nDense lines. Short names.\n')
skills = discover(roots=[root])
skt = {t.__name__: t for t in skill_tools(host, lambda: skills)}
skt['read_skill']('house-style')

'<skill name="house-style" from="/private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp0nwf2kip/proj/.agents/skills/house-style/SKILL.md">\n\nDense lines. Short names.\n\n</skill>'

In [ ]:
test_eq(sorted(skt), ['create_skill', 'read_skill'])
body = skt['read_skill']('house-style')
assert body.startswith('<skill name="house-style"') and 'Dense lines.' in body
test_eq(skt['read_skill']('house'), body)                 # unique prefix resolves to the same skill
assert "no skill matching 'nope'" in skt['read_skill']('nope')
assert 'house-style' in skt['read_skill']('nope')
test_eq({t.__name__ for t in skill_tools(host, lambda: skills) if is_write(t)}, {'create_skill'})

`create_skill` writes under the first open folder in the layout `discover` reads, and refuses rather
than overwriting one that is already there.

In [ ]:
skt['create_skill']('notebook-tests', 'when testing', 'Put the readable case in the notebook.')

'created /private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp0nwf2kip/proj/.agents/skills/notebook-tests/SKILL.md; run /reload to load it into the current agent'

In [ ]:
made = root/'.agents'/'skills'/'notebook-tests'/'SKILL.md'
assert made.exists(), made
assert 'when testing' in made.read_text() and 'readable case' in made.read_text()
assert skt['create_skill']('notebook-tests', 'again', 'body').startswith('refusing to overwrite')
assert 'kebab-case' in skt['create_skill']('Notebook Tests', 'x', 'y')

## Pictures

Image generation uses the current model when it can draw, then a dedicated image endpoint.
`draws_itself` and `from_reply` provide the model-specific operations.

In [ ]:
#| export
RESPONSES_API = 'https://api.openai.com/v1/responses'
IMAGE_API = 'https://api.openai.com/v1/images/generations'
IMAGE_MODEL = 'gpt-image-1'
IMAGE_SIZES = ('1024x1024', '1536x1024', '1024x1536', 'auto')
API_VENDORS = ('openai/', 'azure/')

In [ ]:
#| export
def media_dir(session=''):
    "Return the session's media directory."
    d = Path(session or '.') / 'media'
    d.mkdir(parents=True, exist_ok=True)
    return d

def mime_for(path):
    "Detect a file's MIME type from bytes, then its suffix."
    try: m = detect_mime(Path(path).read_bytes()[:64])
    except OSError: m = None
    return m or mimetypes.guess_type(str(path))[0] or 'image/png'

def save_media(m, session='', stem='image'):
    "Save one media item under the session and return its path."
    mime = m.get('mime') or 'image/png'
    ext = mimetypes.guess_extension(mime) or '.' + mime.split('/')[-1]
    d = media_dir(session)
    n = 1 + len(list(d.glob(f'{stem}-*')))
    p = d / f'{stem}-{n}{ext}'
    p.write_bytes(m['data'])
    return p

def image_available(): return bool(os.environ.get('OPENAI_API_KEY'))

def _hdrs(): return {'Authorization': f"Bearer {os.environ['OPENAI_API_KEY']}"}

def _post_image(prompt, size, n, timeout=120, model=IMAGE_MODEL):
    "Raw `data` rows from the images endpoint."
    import httpx
    r = httpx.post(IMAGE_API, timeout=timeout, headers=_hdrs(), json={'model': model, 'prompt': prompt, 'size': size, 'n': n})
    r.raise_for_status()
    return r.json().get('data') or []

def api_model(model):
    "Strip a supported API-vendor prefix from a model id."
    s = str(model or '')
    for v in API_VENDORS:
        if s.startswith(v): return s[len(v):]
    return s

def _post_responses(prompt, model, timeout=300):
    "Return a Responses API image-generation reply."
    import httpx
    r = httpx.post(RESPONSES_API, timeout=timeout, headers=_hdrs(),
                   json={'model': api_model(model), 'input': prompt, 'tools': [{'type': 'image_generation'}]})
    r.raise_for_status()
    return r.json()

`save_media` chooses the path and name for a generated picture. A frontend can locate the result
from its session.

In [ ]:
png = {'mime': 'image/png', 'data': b'\x89PNG\r\n\x1a\n' + b'0' * 20}
saved = save_media(png, session=root/'s1', stem='generated')
saved, mime_for(saved)

(Path('/private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmp0nwf2kip/proj/s1/media/generated-1.png'),
 'image/png')

In [ ]:
assert saved.exists() and saved.suffix == '.png'
test_eq(saved.parent, media_dir(root/'s1'))
test_eq(saved.read_bytes(), png['data'])
test_eq(mime_for(saved), 'image/png')
assert saved.name.startswith('generated'), saved.name
assert save_media(png, session=root/'s1', stem='generated').name != saved.name

In [ ]:
#| export
def image_tools(host, mx=MAX_TOOL_CHARS, session='', draws_itself=None, from_reply=None, model_id='',
                on_media=None):
    "Drawing: by the turn's own model where it can, and by the images endpoint where it cannot."

    @acts
    @summary(lambda a: f'Draw: {_1(a.get("prompt"), 100)}')
    def generate_image(prompt: str, size: str = '1024x1024', n: int = 1, model: str = '') -> str:
        """Generate and save images.
        `size` is 1024x1024, 1536x1024, 1024x1536, or auto. A non-empty `model` uses the images endpoint.
        """
        if not image_available(): return err('image generation is unavailable', 'OPENAI_API_KEY is not set')
        if size not in IMAGE_SIZES: return err('unknown size', f'{size!r}; use one of {", ".join(IMAGE_SIZES)}')
        own = bool(draws_itself and draws_itself() and from_reply and model_id)
        try:
            if not model and own: media = from_reply(_post_responses(prompt, model_id))
            else: media = [{'mime': 'image/png', 'data': b64decode(r['b64_json'])}
                for r in _post_image(prompt, size, max(1, min(int(n or 1), 4)), model=api_model(model or IMAGE_MODEL)) if r.get('b64_json')]
        except Exception as e: return err('could not generate the image', e)
        if not media: return err('the model returned no image', 'the reply carried no picture')
        try: out = [save_media(m, session, 'generated') for m in media]
        except Exception as e: return err('could not save the image', e)
        if on_media: on_media(out)
        return clip('\n'.join(str(p) for p in out))

    return [generate_image]

The image group is credentialled rather than declared: a key is either there or it is not, and
`tools_for` never probes for one. Without a key the tool says so and reaches nothing.

In [ ]:
image_available(), api_model('gpt-image-1'), api_model('openai/gpt-image-1')

(True, 'gpt-image-1', 'gpt-image-1')

In [ ]:
test_eq(image_available(), bool(os.environ.get('OPENAI_API_KEY')))
test_eq(api_model('openai/gpt-image-1'), 'gpt-image-1')   # a vendor prefix is stripped
test_eq(api_model('gpt-image-1'), 'gpt-image-1')
assert '1024x1024' in IMAGE_SIZES and 'auto' in IMAGE_SIZES
test_eq(IMAGE_MODEL, 'gpt-image-1')
assert RESPONSES_API.startswith('https://') and IMAGE_API.startswith('https://')
assert all(v.endswith('/') for v in API_VENDORS)
gen = image_tools(host)[0]
test_eq(gen.__name__, 'generate_image')
if not image_available():
    assert 'OPENAI_API_KEY' in gen('a kettle')            # it says so rather than reaching the wire
else: assert failed(gen('a kettle', size='3x3'))          # and an unknown size never gets sent

The live image check calls the configured OpenAI image endpoint and verifies that the returned file
is a non-empty image. It is excluded from the default suite because it spends credentials.

In [ ]:
#| eval: false
image_root = Path(tempfile.mkdtemp())
image_host = LocalHost([image_root], index=False)
generate_image = image_tools(image_host, session=image_root)[0]
image_result = generate_image('A simple black circle centered on a white background', n=1)
assert not image_result.startswith('ERROR:'), image_result
image_path = Path(image_result.strip())
assert image_path.exists() and image_path.stat().st_size > 1000
assert mime_for(image_path).startswith('image/')

## Git

Git tools delegate to `gheasy`. A repository found by walking above a supplied path must still be
inside the host's roots.

In [ ]:
#| export
from gheasy.repo import GitRepo, STATE_KEYS, REMOTE_OPS, _said

In [ ]:
#| export
def git_tools(host, mx=MAX_TOOL_CHARS):
    "Git bound to one open repository, kept inside the host's roots."
    def repo(path=''):
        roots = L(host.roots).map(lambda r: Path(r).expanduser().resolve())
        if not roots: raise ValueError('open a project folder first')
        found = GitRepo.at(host.check(path or getattr(host, 'project', '') or roots[0], must_exist=True))
        if not any(found.root.is_relative_to(r) for r in roots):
            raise ValueError(f'Git root {found.root} is outside the open folders; open the repository root first')
        return found
    def state(r, result=''): return {'result': result} | {k: r.info()[k] for k in STATE_KEYS}
    def answer(what, path, make):
        try: return clip(json.dumps(make(repo(path)), indent=2), mx * 2)
        except Exception as e: return err(what, e)
    def preview(r, onto):
        oid = r._resolve_ref(onto)
        stops, replayed, _ = r._replay(oid, list(reversed(r.history(limit=250, ref=f'{oid}..HEAD'))))
        return ({k: v for k, v in r.rebase_preview(onto).items() if k != 'review'}
                | {'stops_at': stops, 'replayed': replayed})
    @summary(lambda a: 'Git status')
    def git_status(path: str = '') -> str:
        "Repository status: branch, upstream, local and remote branches, and changed files."
        return answer('git status', path, state)
    @summary(lambda a: 'Git divergence')
    def git_divergence(path: str = '', upstream: str = '') -> str:
        "How far this branch has run from its upstream each way, and every way back, rehearsed."
        return answer('git divergence', path, lambda r: r.divergence(upstream))
    @summary(lambda a: 'Rehearse a rebase')
    def git_rebase_preview(onto: str, path: str = '') -> str:
        "What replaying this branch onto `onto` would hit, without rewriting anything."
        return answer('git rebase preview', path, lambda r: preview(r, onto))
    @writes
    @summary(lambda a: f'Git {a.get("op","fetch")}')
    def git_remote(op: str = 'fetch', path: str = '') -> str:
        "Talk to the remote: `fetch`, `pull` (fast-forward only), or `push`."
        if op not in REMOTE_OPS: return err('git remote', ValueError(f'op must be one of {", ".join(REMOTE_OPS)}'))
        return answer(f'git {op}', path, lambda r: state(r, _said(getattr(r, op)())))
    @writes
    @summary(lambda a: f'Git checkout {a.get("branch","?")}')
    def git_checkout(branch: str, path: str = '') -> str:
        "Switch to a local branch, or create a local tracking branch from `REMOTE/BRANCH`."
        return answer('git checkout', path, lambda r: state(r, _said(r.checkout(str(branch or '').strip()))))
    return [git_status, git_divergence, git_rebase_preview, git_remote, git_checkout]

A real repository, with one commit and one untracked file.

In [ ]:
repo = Path(tempfile.mkdtemp()).resolve()/'repo'
repo.mkdir(parents=True)
(repo/'a.py').write_text('x = 1\n')
for c in (['init', '-q', '-b', 'main'], ['add', 'a.py'],
          ['-c', 'user.email=t@e', '-c', 'user.name=T', 'commit', '-qm', 'first']):
    subprocess.run(['git', '-C', str(repo)] + c, check=True)
(repo/'b.py').write_text('y = 2\n')
ghost = LocalHost([repo], index=False)
gt = {t.__name__: t for t in git_tools(ghost)}
status = json.loads(gt['git_status']())
status['branch'], status['clean'], [c['path'] for c in status['changes']]

('main', False, ['b.py'])

In [ ]:
test_eq(status['branch'], 'main')
test_eq(status['clean'], False)
test_eq([c['path'] for c in status['changes']], ['b.py'])
test_eq(sorted(gt), sorted(GIT_TOOLS))
test_eq({t.__name__ for t in git_tools(ghost) if is_write(t)}, set(GIT_WRITE_TOOLS))

assert failed(gt['git_divergence']()) and 'tracks nothing' in gt['git_divergence']()

prev = json.loads(gt['git_rebase_preview']('main'))
assert {'current', 'onto', 'clean', 'merge_base'} <= set(prev), sorted(prev)
assert 'stops_at' in prev and 'replayed' in prev, sorted(prev)
test_eq(subprocess.run(['git', '-C', str(repo), 'status', '--porcelain'],
                       capture_output=True, text=True).stdout.strip(), '?? b.py')

assert failed(gt['git_checkout']('no-such-branch'))
assert 'op must be one of' in gt['git_remote']('nonsense')   # never reaches the network
fetched = gt['git_remote']('fetch')                          # no remote, so it is a no-op not a lie
assert failed(fetched) or json.loads(fetched)['branch'] == 'main', fetched

## Which tools a host gets

`Host.provides` selects tool groups. `drop` removes groups from that selection.

In [ ]:
#| export
#: group -> the factory that builds it. Order is the order a model sees the tools in.
GROUPS = (('code', code_tools), ('file', file_tools), ('notebook', notebook_tools),
          ('web', web_tools), ('memory', memory_tools), ('ask', ask_tools),
          ('watch', watch_tools),
          ('api', api_tools), ('session', session_tools), ('shell', shell_tools),
          ('git', git_tools))

def tools_for(host, get_skills=None, extra=(), mx=MAX_TOOL_CHARS, drop=(), image=None):
    """Every tool this host declares it can support, plus whatever else was registered.

    `image` is a built image group, or None. It needs to know what the turn's model can do, and a
    host does not, so the frontend that knows builds it and passes it in.
    """
    drop = set(drop or ())
    tools = [t for g, f in GROUPS if g not in drop and host.can(g) for t in f(host, mx)]
    if get_skills is not None and 'skill' not in drop: tools += skill_tools(host, get_skills, mx)
    if image is not None and 'image' not in drop: tools += list(image)
    return tools + list(extra or ())

`tools_for` maps declared group names to factories. Tests require the table and capability classes to
use the same names.

In [ ]:
[g for g, _ in GROUPS]

['code',
 'file',
 'notebook',
 'web',
 'memory',
 'ask',
 'watch',
 'api',
 'session',
 'shell',
 'git']

In [ ]:
from shalya.host import (ApiHost, AskHost, CodeHost, GitHost, MemoryHost, NotebookHost,
                         SessionHost, ShellHost, WatchHost, WebHost)
declared = {c.group for c in (CodeHost, WebHost, NotebookHost, MemoryHost, AskHost, WatchHost,
                              SessionHost, ShellHost, ApiHost, GitHost)} | {'file'}
test_eq({g for g, _ in GROUPS}, declared)
test_eq(len(GROUPS), len(declared))                       # named once each
assert all(callable(f) for _, f in GROUPS)

A host over a git repository gets ten groups' worth of tools, and the groups it declared it could
not do are simply absent.

In [ ]:
ts = tools_for(ghost)
sorted(t.__name__ for t in ts)

['add_cell',
 'add_root',
 'create_file',
 'edit_cell',
 'edit_file',
 'git_checkout',
 'git_divergence',
 'git_rebase_preview',
 'git_remote',
 'git_status',
 'grep',
 'inspect_python',
 'list_files',
 'list_vars',
 'ls',
 'notebook_cells',
 'outline',
 'read_terminal',
 'read_url',
 'replace_text',
 'research',
 'run_python',
 'run_shell',
 'search_code',
 'similar_code',
 'view_cell',
 'view_file',
 'web_search']

In [ ]:
names = {t.__name__ for t in ts}
assert {'search_code', 'view_file', 'run_shell', 'git_status'} <= names, names
assert not (names & {'memory_search', 'api_load', 'list_watches'}), 'no vault and no specs'
test_eq(names, {t.__name__ for t in tools_for(ghost)})

In [ ]:
fewer = {t.__name__ for t in tools_for(ghost, drop=['shell', 'git'])}
test_eq(names - fewer, {'run_shell', *GIT_TOOLS})
assert 'search_code' in fewer

`file` is not a group a host declares. Every host has the path boundary, so every host gets the file
tools, and the table names it so a budget can still drop it.

## What a sub-agent may have

A sub-agent gets the read tools and, with a budget, a hard cap on how many calls it may make. The
write set comes from the tools themselves: `is_write` reads the mark `@writes` left.

In [ ]:
#| export
def _writing(t): return is_write(t) or getattr(t, '__name__', '') in WRITE_TOOLS
def _acting(t):  return has_effect(t) or getattr(t, '__name__', '') in ACTING_TOOLS

def read_only(tools, max_calls=None, writes=False, effects=True, block=()):
    "The tools an agent may have when it must not act, optionally behind a hard call budget."
    blocked = set(block or ())
    allowed = []
    for t in tools:
        if getattr(t, '__name__', '') in blocked: continue
        if not effects and _acting(t): continue
        if writes: allowed.append(t)
        elif (safe := getattr(t, 'read_only', None)) is not None: allowed.append(safe)
        elif not _writing(t): allowed.append(t)
    if max_calls is None: return allowed
    state, lock = {'n': 0}, threading.Lock()

    def guarded(f):
        @functools.wraps(f)
        def call(*args, **kw):
            with lock:
                state['n'] += 1
                over = state['n'] > max_calls
            if over:
                return ('Tool budget exhausted. Stop calling tools and return the '
                        'best evidence-backed answer now.')
            return f(*args, **kw)
        return call
    return [guarded(t) for t in allowed]

Every write tool is filtered out, whatever else the host offers. `effects=False` withholds the rest
of the ways to act: running code, an API call that can POST, an image that costs money, and standing
work that outlives the turn. None of those writes a file the user owns, so `WRITE_TOOLS` does not
name them and approval does not gate them. An agent asked to propose and not to act is refused them
all the same. What survives can still be pinned: `read_url` keeps its page and loses the vault entry.

In [ ]:
[t.__name__ for t in read_only(ts)]

['search_code',
 'grep',
 'ls',
 'similar_code',
 'outline',
 'list_files',
 'view_file',
 'notebook_cells',
 'view_cell',
 'web_search',
 'read_url',
 'research',
 'list_vars',
 'inspect_python',
 'read_terminal',
 'git_status',
 'git_divergence',
 'git_rebase_preview']

In [ ]:
ro = {t.__name__ for t in read_only(ts)}
test_eq(ro & WRITE_TOOLS, set())
assert 'search_code' in ro and 'git_status' in ro
test_eq({t.__name__ for t in read_only(ts, writes=True)} & {'create_file'}, {'create_file'})
safe_web = {t.__name__: t for t in read_only(wt.values())}
test_eq(wt['read_url'].read_only, safe_web['read_url'])
safe_web['read_url']('https://python.org')
test_eq(web_results.remember, False)
test_eq(set(get_schema(safe_web['read_url'])['input_schema']['properties']), {'url'})
full_web = {t.__name__: t for t in read_only(wt.values(), writes=True)}
full_web['read_url']('https://python.org')
test_eq(web_results.remember, True)

# `effects=False` also withholds what acts without writing a file the user owns
acting = {t.__name__ for t in ts if has_effect(t)}
test_eq(acting - ACTING_TOOLS, set())              # nothing is marked without being named
assert acting <= ro                                # and the default keeps every one of them
propose = {t.__name__ for t in read_only(ts, effects=False)}
test_eq(propose & (WRITE_TOOLS | ACTING_TOOLS), set())
assert 'search_code' in propose                    # looking is still allowed

With a budget, the tools themselves stop the loop. A local engine owns its internal tool loop, so a
cap the harness holds is a cap the harness cannot enforce.

In [ ]:
budgeted = {t.__name__: t for t in read_only(ts, max_calls=1)}
budgeted['search_code']('threshold'), budgeted['ls']('.')

('no matches (Kosha sync in progress; literal fallback via ripgrep)',
 'Tool budget exhausted. Stop calling tools and return the best evidence-backed answer now.')

In [ ]:
budgeted = {t.__name__: t for t in read_only(ts, max_calls=1)}
budgeted['ls']('.')
assert 'budget exhausted' in budgeted['ls']('.'), 'the second call should have been refused'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()